# PDP Validation
##### This code is for validating our pipeline extraction product with Stanford's POLIS database. It reduces the data to the four essential fields for fast and simple comparison: index (number), Name, Size, and Walls. LLM outputs are converted to match the Stanford POLIS formats. Data is paired by row, so validation only occurs for the cities processed by the LLM, if not all 1035 have been processed. The following chunk produces a new dataframe with all mismatches between the datasets, and the outputs lists the count of fields where mismatches are found.
## [Stanford POLIS site](https://polis.stanford.edu)
## [POLIS github database](https://github.com/sdam-au/GI_ETL/blob/master/data/polis_database.csv)

# 0. Setup

In [1]:
import pandas as pd
import csv

In [2]:
# Load data
df_llm = pd.read_csv('poleis_llm.csv')
df_db = pd.read_csv('polis_database.csv')

# column name congruency
df_llm_selected = df_llm[['polis_id', 'territory_size', 'walls']].copy().rename(
    columns={"polis_id": 'Name', 'territory_size': 'Size', 'walls': 'Walls'})
df_db_selected = df_db[['name', 'Size', 'Walls']].copy().rename(
    columns={'name': 'Name'})

# Create new column for numerial index for easier sorting
df_llm_selected.insert(0, 'index', df_llm_selected['Name'].str.extract(r'(\d+)').astype(int))
df_db_selected.insert(0, 'index', df_db_selected['Name'].str.extract(r'(\d+)').astype(int))

# Walls: True -> 1.0, False -> 0.0, NaN -> 0.0
df_llm_selected['Walls'] = df_llm_selected['Walls'].map({True: 1.0, False: 0.0}).fillna(0.0)
# Size: NaN -> 0.0
df_llm_selected['Size'] = df_llm_selected['Size'].fillna(0.0)

# Sort by index
df_llm_selected = df_llm_selected.sort_values('index').reset_index(drop=True)
df_db_selected = df_db_selected.sort_values('index').reset_index(drop=True)

# Limit validation to those in llm set
df_db_selected = df_db_selected[df_db_selected['index'].isin(df_llm_selected['index'])]

# Save to new .csv
df_llm_selected.to_csv('val_llm.csv', index=False)
df_db_selected.to_csv('val_stanford.csv', index=False)

# Preview 
print(df_llm_selected.head())
print(df_db_selected.head())

   index         Name  Size  Walls
0      1     1-Alalie   3.0    1.0
1      2   2-Emporion   2.0    1.0
2      3   3-Massalia   2.0    1.0
3      4      4-Rhode   0.0    0.0
4      5  5-Abakainon   0.0    1.0
   index         Name  Size  Walls
0      1     1-Alalie   3.0    1.0
1      2   2-Emporion   2.0    1.0
2      3   3-Massalia   2.0    1.0
3      4      4-Rhode   0.0    0.0
4      5  5-Abakainon   0.0    1.0


# Find mismatches
For each mismatch found, the llm row is listed, followed by the stanford row

In [6]:

def find_mismatches(llm_file='val_llm.csv', stanford_file='val_stanford.csv'):
    # Load data
    llm_df = pd.read_csv(llm_file)
    stanford_df = pd.read_csv(stanford_file)
    
    mismatches = []
    # Make counters
    name_mismatches = 0
    size_mismatches = 0
    walls_mismatches = 0
    
    # Use the minimum length to avoid out-of-bounds errors
    limit = min(len(llm_df), len(stanford_df))
    
    for idx in range(limit):
        llm_row = llm_df.iloc[idx]
        stanford_row = stanford_df.iloc[idx]
        
        # Check specific columns
        # We cast to str to ensure '1.0' vs 1.0 doesn't trigger a false mismatch
        is_size_diff = str(llm_row['Size']) != str(stanford_row['Size'])
        is_walls_diff = str(llm_row['Walls']) != str(stanford_row['Walls'])
        is_name_diff = str(llm_row['Name']) != str(stanford_row['Name'])
        
        if is_size_diff or is_walls_diff or is_name_diff:
            # Update specific counters
            if is_size_diff: size_mismatches += 1
            if is_walls_diff: walls_mismatches += 1
            if is_name_diff: name_mismatches += 1
            
            # Store the pair for the CSV
            mismatches.append(llm_row.to_dict())
            mismatches.append(stanford_row.to_dict())
    
    # Create and save CSV
    mismatches_df = pd.DataFrame(mismatches)
    mismatches_df.to_csv('val_mismatches.csv', index=False)
    
    # Print detailed report
    print(f"Total Rows Compared: {limit}")
    print(f"Rows with any Error: {len(mismatches) // 2}")
    print("-" * 25)
    print(f"Name Mismatches:     {name_mismatches}")
    print(f"Size Mismatches:     {size_mismatches}")
    print(f"Walls Mismatches:    {walls_mismatches}")
    print("-" * 25)
    print(f"Mismatching rows saved to 'val_mismatches.csv'")
    
    return mismatches_df

find_mismatches()

Total Rows Compared: 382
Rows with any Error: 111
-------------------------
Name Mismatches:     29
Size Mismatches:     63
Walls Mismatches:    37
-------------------------
Mismatching rows saved to 'val_mismatches.csv'


,index,Name,Size,Walls
0,8,8-Aitna,0.0,0.0
1,8,8-Aitna,4.0,0.0
2,11,11-Alaisa,0.0,1.0
3,11,11-Alaisa,0.0,0.0
4,12,12-Alontion,0.0,1.0
...,...,...,...,...
217,372,372-Histiaia/Oreos,4.0,1.0
218,376,376-Posideion,0.0,0.0
219,376,376-Posideion,1.0,0.0
220,377,377-Styra,0.0,1.0
